# Interfaccia per gli strumenti

In [34]:
import pyvisa
import numpy as np
import time
import serial
from serial.tools import list_ports
import matplotlib.pyplot as plt
import cv2

import ipywidgets as widgets
from lib.drivers import *
import threading

In [35]:
%matplotlib widget
plt.close('all')

In [36]:
# elenca porte seriali disponibili
ports = list_ports.comports()
usb_ports = [port for port in ports if "usb" in port.description.lower()]
for port in usb_ports:
    print(port.device, port.description)

In [ ]:
class BaseLabWidget(widgets.VBox):
    def __init__(self, instruments, acquire_hz=10, render_hz=5, window=100, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.instruments = instruments
        self.acquire_interval = 1.0 / acquire_hz
        self.render_interval  = 1.0 / render_hz
        self.running = False
        self.data  = {k: [] for k in instruments}
        self.times = {k: [] for k in instruments}
        self._lock = threading.Lock()

        self.start_btn = widgets.Button(description="Start")
        self.stop_btn  = widgets.Button(description="Stop")
        self.start_btn.on_click(self.start)
        self.stop_btn.on_click(self.stop)
        self.plot_output = widgets.Output()

